# Blood plasma proteome — FASTA builder

Reads protein group files from Geyer et al. (2019), queries UniProt for sequences and metadata (Swiss-Prot reviewed entries only), and writes a FASTA file to `export/`.

In [ ]:
import os
import time
import pandas as pd
import numpy as np
from blood_proteome.fn import (
    get_swissprot_sequences_batch,
    query_human_proteome,
    create_human_proteome_fasta,
    validate_fasta,
    WEBSITE_API,
    get_url,
)

## Load plasma proteome data

Geyer et al. (2019) — serum vs plasma cohort.

In [ ]:
geyer_serum  = pd.read_csv('Files/SerumVsPlasma_txt/proteinGroups.txt', sep='\t', low_memory=False)
geyer_plasma = pd.read_csv('Files/Plasma_20Ind_combined/proteinGroups.txt', sep='\t', low_memory=False)

## Extract unique protein IDs

In [ ]:
geyer_serum_protein  = geyer_serum['Majority protein IDs'].str.split(';').explode().unique()
geyer_plasma_protein = geyer_plasma['Majority protein IDs'].str.split(';').explode().unique()
print(f'Serum proteins:  {len(geyer_serum_protein)}')
print(f'Plasma proteins: {len(geyer_plasma_protein)}')

In [ ]:
trial_ids = [
    pid.strip() for pid in geyer_plasma_protein
    if pid and pid.strip()
]
print(f'Unique protein IDs to query: {len(trial_ids)}')
print(f'Sample: {trial_ids[:5]}')

## Retrieve sequences from UniProt

Fetches Swiss-Prot reviewed entries only.

In [ ]:
protein_sequences = get_swissprot_sequences_batch(trial_ids)
print(f'\nRetrieved: {len(protein_sequences)} sequences')

## Export FASTA

In [ ]:
import time, os

today = time.strftime('%y%m%d')
os.makedirs('export', exist_ok=True)
fasta_filename = f'export/Blood_proteome_{today}.fasta'

def build_header(entry):
    prot_name = 'Unknown protein'
    desc = entry.get('proteinDescription', {})
    if 'recommendedName' in desc and desc['recommendedName']:
        fn = desc['recommendedName'].get('fullName', '')
        prot_name = fn.get('value', prot_name) if isinstance(fn, dict) else fn or prot_name
    elif 'submittedName' in desc and desc['submittedName']:
        submitted = desc['submittedName']
        if isinstance(submitted, list) and submitted:
            fn = submitted[0].get('fullName', '')
            prot_name = fn.get('value', prot_name) if isinstance(fn, dict) else fn or prot_name

    accession  = entry.get('primaryAccession', 'UNKNOWN')
    entry_name = str(entry.get('uniProtkbId') or entry.get('uniProtKBId') or f'{accession}_HUMAN')
    reviewed   = entry.get('reviewed', None)
    is_reviewed = reviewed if isinstance(reviewed, bool) else 'Swiss-Prot' in entry.get('entryType', '')
    db_type = 'sp' if is_reviewed else 'tr'

    gene_name = ''
    genes = entry.get('genes', [])
    if isinstance(genes, list) and genes:
        gi = genes[0]
        if isinstance(gi, dict):
            gene_name = gi.get('geneName', {}).get('value', '') or gi.get('geneName', '')

    taxid = entry.get('organism', {}).get('taxonId') if isinstance(entry.get('organism'), dict) else None
    pe = '1' if is_reviewed else '2'
    sv = str(entry.get('version', 1))
    parts = [f'{db_type}|{accession}|{entry_name}', prot_name.strip(),
             'OS=Homo sapiens', f'OX={taxid or 9606}']
    if gene_name:
        parts.append(f'GN={gene_name}')
    parts += [f'PE={pe}', f'SV={sv}']
    return ' '.join(parts)


skipped = 0
with open(fasta_filename, 'w') as fasta_file:
    for entry in protein_sequences:
        sequence = entry.get('sequence', {}).get('value', '')
        if not sequence:
            skipped += 1
            continue
        fasta_file.write(f'>{build_header(entry)}\n')
        for i in range(0, len(sequence), 60):
            fasta_file.write(sequence[i:i + 60] + '\n')

file_size = os.path.getsize(fasta_filename)
print(f'FASTA written: {fasta_filename}')
print(f'Proteins: {len(protein_sequences) - skipped}  |  Skipped (no sequence): {skipped}')
print(f'Size: {file_size / (1024 * 1024):.1f} MB')

## Validate output

In [ ]:
validate_fasta(fasta_filename)

## Optional — full human proteome reference

> **Note:** downloads ~20 000 proteins and may take several minutes.

```python
# Swiss-Prot only (~20 K proteins, recommended)
create_human_proteome_fasta(reviewed_only=True,  file_loc='export/')

# All human proteins including TrEMBL (~180 K proteins)
create_human_proteome_fasta(reviewed_only=False, file_loc='export/')
```